# 🌿 DeepWeeds — Google Colab Runner

Notebook này thực hiện:
1. **Clone repo** từ GitHub
2. **Cài đặt** dependencies
3. **Download + giải nén** data từ Zenodo
4. **Chạy** toàn bộ thí nghiệm 5 bước
5. **Nén** kết quả và **download** về máy local

> ⚠️ **Trước khi chạy**: Vào `Runtime → Change runtime type → GPU (T4)`

## 📦 CELL 1 — Clone repo & Setup

In [ ]:
import os, sys

# ⚠️ SỬA CHỖ NÀY: URL repo GitHub của bạn
REPO_URL = 'https://github.com/YOUR_USERNAME/K4-Track4-Day2-Deeplearning-Advance.git'
REPO_DIR = '/content/deepweeds'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    print('✅ Repo đã tồn tại')

os.chdir(REPO_DIR)
for p in [REPO_DIR, os.path.join(REPO_DIR, 'code')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'📁 CWD: {os.getcwd()}')


## 🔧 CELL 2 — Cài đặt dependencies

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'timm', 'albumentations', 'openpyxl'])

import torch
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory/1024**3:.1f} GB')
else:
    print('⚠️ Không tìm thấy GPU!')


## 📥 CELL 3 — Download & giải nén Data

In [ ]:
import os, zipfile, hashlib
from pathlib import Path

DATA_DIR   = Path('data')
IMAGES_DIR = DATA_DIR / 'images'
LABELS_DIR = DATA_DIR / 'labels'
IMAGES_ZIP = DATA_DIR / 'images.zip'

for d in [DATA_DIR, IMAGES_DIR, LABELS_DIR]:
    d.mkdir(exist_ok=True)

# --- Download images.zip từ Zenodo ---
IMAGES_URL = 'https://zenodo.org/records/7939060/files/images.zip?download=1'
IMAGES_MD5 = 'b7b30f96d466fba86016aa5a26606e0f'

def md5_check(path, expected):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest() == expected

if not IMAGES_ZIP.exists():
    print('⬇️  Download images.zip (~470MB)...')
    os.system(f'wget -q --show-progress -O "{IMAGES_ZIP}" "{IMAGES_URL}"')
else:
    print('✅ images.zip đã có')

if md5_check(IMAGES_ZIP, IMAGES_MD5):
    print('✅ MD5 hợp lệ')
else:
    print('❌ MD5 KHÔNG KHỚP!')

# --- Giải nén ---
img_count = len(list(IMAGES_DIR.glob('*.jpg')))
if img_count < 17000:
    print(f'📂 Giải nén ({img_count} ảnh hiện có)...')
    with zipfile.ZipFile(IMAGES_ZIP, 'r') as zf:
        for member in zf.namelist():
            fname = Path(member).name
            if fname.endswith('.jpg') and fname:
                with zf.open(member) as src, open(IMAGES_DIR/fname, 'wb') as dst:
                    dst.write(src.read())
    print(f'✅ Giải nén xong! {len(list(IMAGES_DIR.glob("*.jpg")))} ảnh')
else:
    print(f'✅ Ảnh đã có: {img_count}')

# --- Download labels từ GitHub DeepWeeds ---
BASE = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
FILES = ['labels.csv'] + [
    f'{split}_subset{i}.csv'
    for i in range(5) for split in ['train', 'val', 'test']
]
print('\n⬇️  Download labels...')
for fname in FILES:
    dest = LABELS_DIR / fname
    if not dest.exists():
        os.system(f'wget -q -O "{dest}" "{BASE}/{fname}"')
        print(f'  ✅ {fname}')
    else:
        print(f'  ⏭️  {fname} (đã có)')

print('\n🎉 Data setup xong!')


## ✅ CELL 4 — Kiểm tra dữ liệu

In [ ]:
import dataset as ds
import pandas as pd

FOLD = 0
train_df, val_df, test_df = ds.load_split('data/labels', fold=FOLD)
stats = ds.check_split(train_df, val_df, test_df, 'data/images')

print('=== KIỂM TRA FOLD', FOLD, '===')
print(f"Train: {stats['n_train']} | Val: {stats['n_val']} | Test: {stats['n_test']}")
print(f"Classes: {stats['n_classes']}")
print(f"Ảnh tìm thấy: {stats['n_images_found']} / {stats['n_total']}")


## ⚙️ CELL 5 — Cấu hình Colab

In [ ]:
import os, torch
import train as tr
from train import Config

# Colab T4: batch lớn hơn, num_workers > 0
COLAB_DEFAULTS = dict(
    images_dir  = 'data/images',
    labels_dir  = 'data/labels',
    out_dir     = 'runs',
    pred_dir    = 'predictions',
    curves_dir  = 'curves',
    num_workers = 2,
    amp         = True,
    batch_size  = 64,
    epochs      = 20,
    fold        = 0,
)

for d in ['runs', 'predictions', 'curves']:
    os.makedirs(d, exist_ok=True)

print('✅ Config sẵn sàng: batch=64, workers=2, epochs=20, AMP=True')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')


## 🔬 CELL 6 — BƯỚC 1: Backbone × Init

In [ ]:
import pandas as pd
step1_results = []

BACKBONES = ['resnet34', 'resnet50', 'convnext_tiny']
INITS     = ['finetune', 'frozen', 'scratch']

for backbone in BACKBONES:
    for init in INITS:
        exp_id = f'B_{backbone}_{init}'
        print(f'\n>>> {exp_id}')
        cfg = Config(exp_id=exp_id, seed=0, backbone=backbone, init=init,
                     img_size=224, **COLAB_DEFAULTS)
        torch.cuda.empty_cache()
        res = tr.run(cfg)
        step1_results.append(dict(exp_id=exp_id, backbone=backbone, init=init,
                                  best_val_f1=res['best_val_f1'], best_epoch=res['best_epoch']))
        print(f'✅ Val F1={res["best_val_f1"]:.4f}')

df1 = pd.DataFrame(step1_results).sort_values('best_val_f1', ascending=False)
df1.to_csv('runs/step1_summary.csv', index=False)
print('\n📊 Bước 1:')
print(df1.to_string(index=False))

BEST_BACKBONE = df1.iloc[0]['backbone']
BEST_INIT     = df1.iloc[0]['init']
print(f'\n🏆 Best: {BEST_BACKBONE} + {BEST_INIT}')


## 🔬 CELL 7 — BƯỚC 2: Augmentation × Loss

In [ ]:
# BEST_BACKBONE = 'resnet34'; BEST_INIT = 'finetune'  # uncomment nếu chạy độc lập
step2_results = []

for aug in ['basic', 'color', 'trivial', 'randaug']:
    for loss in ['ce', 'ls', 'focal', 'ce_weighted']:
        exp_id = f'T_{aug}_{loss}'
        print(f'>>> {exp_id}')
        cfg = Config(exp_id=exp_id, seed=0, backbone=BEST_BACKBONE, init=BEST_INIT,
                     aug=aug, loss=loss,
                     label_smoothing=0.1 if loss=='ls' else 0.0,
                     class_weight_beta=0.9999 if loss=='ce_weighted' else None,
                     **COLAB_DEFAULTS)
        torch.cuda.empty_cache()
        res = tr.run(cfg)
        step2_results.append(dict(exp_id=exp_id, aug=aug, loss=loss,
                                  best_val_f1=res['best_val_f1']))
        print(f'✅ Val F1={res["best_val_f1"]:.4f}')

df2 = pd.DataFrame(step2_results).sort_values('best_val_f1', ascending=False)
df2.to_csv('runs/step2_summary.csv', index=False)
print('\n📊 Bước 2:')
print(df2.to_string(index=False))

BEST_AUG  = df2.iloc[0]['aug']
BEST_LOSS = df2.iloc[0]['loss']
print(f'🏆 Best: aug={BEST_AUG}, loss={BEST_LOSS}')


## 🔬 CELL 8 — BƯỚC 3: Mixup / CutMix + Sampler

In [ ]:
step3_results = []
experiments = [
    dict(mix=None,     sampler=None,        name='baseline'),
    dict(mix='mixup',  sampler=None,        name='mixup'),
    dict(mix='cutmix', sampler=None,        name='cutmix'),
    dict(mix='mixup',  sampler='balanced',  name='mixup_bal'),
    dict(mix='cutmix', sampler='balanced',  name='cutmix_bal'),
]

for exp in experiments:
    exp_id = f'R_{exp["name"]}'
    print(f'>>> {exp_id}')
    cfg = Config(exp_id=exp_id, seed=0, backbone=BEST_BACKBONE, init=BEST_INIT,
                 aug=BEST_AUG, loss=BEST_LOSS,
                 mix=exp['mix'], sampler=exp['sampler'],
                 **COLAB_DEFAULTS)
    torch.cuda.empty_cache()
    res = tr.run(cfg)
    step3_results.append(dict(exp_id=exp_id, **exp, best_val_f1=res['best_val_f1']))
    print(f'✅ Val F1={res["best_val_f1"]:.4f}')

df3 = pd.DataFrame(step3_results).sort_values('best_val_f1', ascending=False)
df3.to_csv('runs/step3_summary.csv', index=False)
print('\n📊 Bước 3:')
print(df3.to_string(index=False))

BEST_MIX     = df3.iloc[0]['mix']
BEST_SAMPLER = df3.iloc[0]['sampler']
print(f'🏆 Best: mix={BEST_MIX}, sampler={BEST_SAMPLER}')


## 🔬 CELL 9 — BƯỚC 4: Final Multi-seed + Test Predictions

In [ ]:
import numpy as np
step4_results = []

for seed in [0, 1, 2]:
    exp_id = f'F_seed{seed}'
    print(f'\n>>> {exp_id}')
    cfg = Config(exp_id=exp_id, seed=seed,
                 backbone=BEST_BACKBONE, init=BEST_INIT,
                 aug=BEST_AUG, loss=BEST_LOSS,
                 mix=BEST_MIX, sampler=BEST_SAMPLER,
                 save_test_predictions=True,
                 ema_decay=0.9998,
                 **COLAB_DEFAULTS)
    torch.cuda.empty_cache()
    res = tr.run(cfg)
    tm = res['test_metrics']
    step4_results.append(dict(
        exp_id=exp_id, seed=seed,
        best_val_f1=res['best_val_f1'],
        test_f1=tm['macro_f1'] if tm else None,
        test_acc=tm['top1'] if tm else None,
    ))
    print(f'✅ Val F1={res["best_val_f1"]:.4f}')

df4 = pd.DataFrame(step4_results)
df4.to_csv('runs/step4_summary.csv', index=False)
vals = df4['best_val_f1'].values
print(f'\n📊 Val Macro-F1: {vals.mean():.4f} ± {vals.std():.4f}')
print(df4.to_string(index=False))


## 🔬 CELL 10 — BƯỚC 5: Inference (TTA + Ensemble)

In [ ]:
import inference as inf_module
import eval as ev
import numpy as np

def load_probs(path):
    df = pd.read_csv(path)
    prob_cols = [c for c in df.columns if c.startswith('prob_')]
    return df['y_true'].values, df[prob_cols].values

# Ensemble val predictions từ 3 seeds
seeds = [0, 1, 2]
val_probs_list, test_probs_list = [], []
y_val_true = None

for seed in seeds:
    vp = f'predictions/F_seed{seed}_seed{seed}_val.csv'
    tp = f'predictions/F_seed{seed}_seed{seed}_test.csv'
    if os.path.exists(vp):
        y_t, probs = load_probs(vp)
        val_probs_list.append(probs)
        y_val_true = y_t
    if os.path.exists(tp):
        _, probs = load_probs(tp)
        test_probs_list.append(probs)

if val_probs_list:
    ens_val_probs = np.mean(val_probs_list, axis=0)
    m = ev.compute_metrics(y_val_true, ens_val_probs.argmax(1), ens_val_probs)
    print(f'📊 Ensemble Val Macro-F1 : {m["macro_f1"]:.4f}')
    print(f'   Ensemble Val Accuracy : {m["top1"]:.4f}')
    print(f'   Ensemble Val ECE      : {m["ece"]:.4f}')
else:
    print('⚠️ Chưa có prediction files. Chạy BƯỚC 4 trước.')

print('\n✅ BƯỚC 5 hoàn tất!')


## 📊 CELL 11 — Tổng hợp & Biểu đồ

In [ ]:
import matplotlib.pyplot as plt
import os

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DeepWeeds — Kết quả thí nghiệm', fontsize=14, fontweight='bold')

for ax, (csv, title, color) in zip(axes, [
    ('runs/step1_summary.csv', 'Bước 1: Backbone & Init', 'steelblue'),
    ('runs/step2_summary.csv', 'Bước 2: Aug & Loss',      'darkorange'),
    ('runs/step4_summary.csv', 'Bước 4: Final (3 seed)', 'seagreen'),
]):
    if not os.path.exists(csv):
        ax.set_title(f'{title}\n(chưa có data)')
        continue
    df = pd.read_csv(csv)
    col = 'best_val_f1'
    bars = ax.bar(df['exp_id'], df[col], color=color)
    ax.set_title(title); ax.set_ylabel('Val Macro-F1')
    ax.tick_params(axis='x', rotation=45); ax.set_ylim(0, 1.0)
    for bar, v in zip(bars, df[col]):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}',
                ha='center', fontsize=7)

plt.tight_layout()
plt.savefig('curves/summary_all_steps.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Biểu đồ lưu tại curves/summary_all_steps.png')


## 💾 CELL 12 — Nén & Download kết quả về máy

In [ ]:
import zipfile, os
from pathlib import Path
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
zip_name  = f'deepweeds_results_{timestamp}.zip'

# Bao gồm (KHÔNG bao gồm data/ và images.zip để tránh file quá lớn)
INCLUDE = ['runs/', 'predictions/', 'curves/', 'code/', 'eval.py',
           'Lab_Day2_DeepWeeds.ipynb', 'Colab_DeepWeeds.ipynb', 'README.md', 'RUBRIC.md']
SKIP_EXT  = set()  # Đặt {'.pth'} nếu muốn bỏ model weights để nhỏ hơn
SKIP_DIRS = {'__pycache__', '.git', '.ipynb_checkpoints'}

total = 0
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in INCLUDE:
        p = Path(item)
        if p.is_dir():
            for f in p.rglob('*'):
                if f.is_file() and f.suffix not in SKIP_EXT:
                    if not any(s in f.parts for s in SKIP_DIRS):
                        zf.write(f); total += 1
        elif p.is_file():
            zf.write(p); total += 1

size_mb = Path(zip_name).stat().st_size / 1024**2
print(f'✅ Đã nén {total} files | {size_mb:.1f} MB → {zip_name}')

try:
    from google.colab import files
    files.download(zip_name)
    print('⬇️  Download bắt đầu...')
except ImportError:
    print(f'ℹ️  Lưu tại: {os.getcwd()}/{zip_name}')


## 💾 CELL 13 — (Tùy chọn) Nén TOÀN BỘ kể cả model .pth

> Lưu ý: file có thể nặng vài GB

In [ ]:
import zipfile, os
from pathlib import Path
from datetime import datetime
from google.colab import files

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
zip_full  = f'deepweeds_FULL_{timestamp}.zip'
SKIP_DIRS = {'data', '.git', '__pycache__', '.ipynb_checkpoints'}
SKIP_FILES = {'images.zip'}

total = 0
with zipfile.ZipFile(zip_full, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in Path('.').rglob('*'):
        if f.is_file() and f.name not in SKIP_FILES:
            if not any(s in f.parts for s in SKIP_DIRS):
                zf.write(f); total += 1

size_mb = Path(zip_full).stat().st_size / 1024**2
print(f'✅ {total} files | {size_mb:.1f} MB')
files.download(zip_full)
